# Compare purchase-history groups

Do the overall results hide differences between customers?

We chose three exploratory groups before inspecting their scores: 1–5, 6–15 and 16+ eligible past orders. The count stops before the hidden validation basket. These groups don't include customers with no purchase history.

The code below reads the saved results; it doesn't retrain a method or score final baskets.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / '03_Code/01_Data_Preparation/prepare_data.py').exists())
report = json.loads((ROOT / '05_Results/03_Checks/history_group_validation.json').read_text(encoding='utf-8'))
groups = pd.DataFrame(report['groups'])
pd.DataFrame({
    'History group': groups['group'],
    'Customers': groups['customers'],
    'Popularity recall (%)': groups['popular_recall_at_5'] * 100,
    'Personal recall (%)': groups['personal_recall_at_5'] * 100,
    'Mean basket size': groups['mean_actual_basket_products'],
    'Customers with short suggestion lists': groups['customers_with_fewer_than_five_suggestions']
}).round(2)

## Read the table together
Recall is the share of each actual basket matched, averaged over customers in that group. Group customer counts are shown so we don't treat differently sized groups as equal samples.

Which group has the highest personal recall? Does the result support saying more history always means higher recall?

These groups can also differ in basket size and shopping habits. The table doesn't isolate a causal effect of history length. We'll discuss the interpretation before writing business findings.

## Optional: reproduce
Uses saved customer-level validation outputs plus the prepared order history. Checks all customers are assigned once and grouped scores recover the overall scores.

In [ ]:
import runpy
analysis = runpy.run_path(str(ROOT / '03_Code/03_Model_Comparison/compare_history_groups.py'))
report = analysis['compare_groups']()

# Why products were missed

We split each hidden validation basket into matched products, missed familiar products and missed new-to-history products. Familiar means present before that customer's validation cutoff. It doesn't use the full prior history or assume lifetime purchases.

The diagnostic uses the hidden baskets only after predictions are made. Don't turn its answers into model features.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / '03_Code/01_Data_Preparation/prepare_data.py').exists())
report = json.loads((ROOT / '05_Results/03_Checks/validation_miss_diagnosis.json').read_text())
pd.DataFrame([
    {'Miss type': 'Familiar product, not suggested', 'Product lines': report['known_product_misses'], 'Share of misses (%)': report['pct_of_missed_lines_known']},
    {'Miss type': 'Not in eligible purchase history', 'Product lines': report['new_product_misses'], 'Share of misses (%)': report['pct_of_missed_lines_new']}
]).round(2)

## What the denominator means
These percentages divide by **all missed product lines**, pooled across 131,209 validation baskets. They aren't customer percentages or the average per-customer recall.

## Familiar misses aren't all fixable by better ranking
A list has only five slots. If a basket contains six familiar products, even a perfect five-product list misses at least one. The report separates this capacity limit from the maximum misses a perfect rebuy ranking could recover. That upper bound uses hidden answers; it isn't a real algorithm or a promised improvement.

## Discuss together
Could changing the ranking of previously bought products ever suggest a product absent from that customer's history?

No changes to the recommendation method have been made from this diagnostic. Final baskets remain unscored.

In [ ]:
pd.DataFrame([
    {'Familiar miss type': 'Beyond five-slot capacity', 'Product lines': report['known_misses_beyond_five_slot_capacity']},
    {'Familiar miss type': 'Recoverable only with perfect rebuy ranking', 'Product lines': report['known_misses_recoverable_with_perfect_ranking']}
])

## Optional: reproduce

In [ ]:
import runpy
analysis = runpy.run_path(str(ROOT / '03_Code/03_Model_Comparison/classify_validation_misses.py'))
report, customer_diagnosis = analysis['classify_misses']()

# Does recency help every history group?

Compare the fixed five-order half-life rule with personal frequency on the same saved validation baskets. History means eligible orders before the target, not total lifetime orders. Groups are the same 1–5, 6–15 and 16+ used earlier.

In [ ]:
# Load saved comparison results; no model needs to be rebuilt.
from pathlib import Path
import json
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / '03_Code/01_Data_Preparation/prepare_data.py').exists())
report = json.loads((ROOT / '05_Results/03_Checks/recency_group_validation.json').read_text())
pd.DataFrame([
    {'History': g['group'], 'Customers': g['customers'],
     'Frequency recall (%)': g['recall_at_5']['frequency'] * 100,
     'Recency recall (%)': g['recall_at_5']['recency'] * 100,
     'Difference (percentage points)': g['recall_at_5']['difference_pp']}
    for g in report['groups']
]).round(2)

## Read the result
Mean recall improves by about 0.02 percentage points with short history, 0.69 with medium history and 2.88 with long history. The average gain is concentrated in the long-history group. This doesn't establish why it happens or prove preferences changed.

**A better group average doesn't mean everyone benefits.** The next table shows customers with more, equal or fewer matches compared with frequency. We are comparing historical predictions, not customers accepting or rejecting suggestions.

In [ ]:
# Show individual wins and losses so averages don't hide them.
pd.DataFrame([
    {'History': g['group'], 'More matches': g['customers_with_more_matches'],
     'Same matches': g['customers_with_equal_matches'],
     'Fewer matches': g['customers_with_fewer_matches']}
    for g in report['groups']
])

## Limits
The short group's hit rate actually changes slightly downward (73.59% to 73.58%), even though its mean recall changes slightly upward. Different metrics can move differently. Approximate paired intervals are in the saved report; subgroup analysis is exploratory, not evidence of causal effects of history length. The final evaluation baskets remain unscored.

## Optional: reproduce

In [ ]:
# Recalculate group summaries from saved validation scores.
import runpy
analysis = runpy.run_path(str(ROOT / '03_Code/03_Model_Comparison/compare_recency_groups.py'))
report = analysis['compare_recency_groups']()

## Boosted trees versus weighted recency: individual customers

This reads checked saved outcomes, without fitting or importing DuckDB. Each customer is counted once on the same validation basket. More/equal/fewer refers to matching suggestions, not customer satisfaction; equal match counts can come from different lists. Source: compare_recency_groups.py --trees.


In [ ]:
# Open the checked individual-outcome report without rerunning the comparison.
from pathlib import Path
import json
import pandas as pd
project_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / '02_Data/02_Prepared/evaluation_plan.csv').exists())
tradeoffs = json.loads((project_root / '05_Results/03_Checks/tree_customer_tradeoffs.json').read_text(encoding='utf-8'))
assert tradeoffs['status'] == 'passed'
rows = [{'Outcome': label, 'Customers': tradeoffs['overall'][key]['customers'],
         'Share (%)': tradeoffs['overall'][key]['pct_of_group']}
        for key, label in [('more_matches', 'More matches'), ('same_matches', 'Same match count'), ('fewer_matches', 'Fewer matches')]]
display(pd.DataFrame(rows))


## Matched local runtime: read the saved measurements

Separate warm local stages for90customers,20repeats each; not full app response time. Identical prepared candidates. Raw feature construction, startup, network and rendering excluded. Cache is small and shared by both methods. This cell reads saved evidence without rerunning timings or training.


In [ ]:
# Read saved timing measurements without rerunning the benchmark.
from pathlib import Path
import json
import pandas as pd
project_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / '02_Data/02_Prepared/evaluation_plan.csv').exists())
runtime = json.loads((project_root / '05_Results/03_Checks/prediction_runtime.json').read_text(encoding='utf-8'))
assert runtime['status'] == 'passed'
rows = [{'Stage': 'Prepared feature fetch/format', **runtime['feature_fetch_and_format']}]
for stage, key in [('Scoring/ranking', 'scoring_and_ranking'), ('Saved-list fetch', 'cached_list_retrieval')]:
    rows.extend({'Stage': stage + ': ' + method, **values} for method, values in runtime[key].items())
display(pd.DataFrame(rows)[['Stage', 'median_ms', 'p95_ms', 'observations']])


## Product coverage and concentration: saved comparison

Compare all three methods on the same validation cohort. Coverage divides distinct suggested products by the union of eligible history candidates. Concentration divides slots occupied by each method's own ten leaders by all filled slots. Short lists reflect candidate limits. This reads saved evidence; broader coverage isn't automatically better for Buy again.


In [ ]:
# Read the checked coverage report, without training or rebuilding recommendations.
from pathlib import Path
import json
import pandas as pd
project_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / '02_Data/02_Prepared/evaluation_plan.csv').exists())
coverage = json.loads((project_root / '05_Results/03_Checks/product_coverage_validation.json').read_text(encoding='utf-8'))
assert coverage['status'] == 'passed'
rows = [{'Method': method, 'Distinct products': values['distinct_products_suggested'],
         'Eligible-product coverage (%)': values['pct_of_eligible_product_pool_suggested'],
         'Top-ten concentration (%)': values['top10_share_of_suggestion_slots_pct'],
         'Customers with short lists': values['customers_with_fewer_than_five']}
        for method, values in coverage['methods'].items()]
display(pd.DataFrame(rows))


## Actual basket size: saved validation comparison

These groups refer to target basket product count, not earlier orders. Only diagnostic scoring sees actual size. Five slots limit recall in bigger baskets, while precision/hit can increase. This cell reads saved metrics for all three methods; it doesn't fit or score again.


In [ ]:
# Read saved basket-size diagnostics without rebuilding predictions.
from pathlib import Path
import json
import pandas as pd
project_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / '02_Data/02_Prepared/evaluation_plan.csv').exists())
basket_report = json.loads((project_root / '05_Results/03_Checks/basket_size_validation.json').read_text(encoding='utf-8'))
assert basket_report['status'] == 'passed'
rows = [{'Actual basket': group['actual_basket_size'], 'Customers': group['customers'], 'Method': method,
         'Precision (%)': 100 * metrics['precision_at_5'], 'Recall (%)': 100 * metrics['recall_at_5'],
         'Hit rate (%)': 100 * metrics['hit_at_5']}
        for group in basket_report['groups'] for method, metrics in group['methods'].items()]
display(pd.DataFrame(rows))


## Final predictions: saved before final answers

This section reads the prediction-only preparation report. It has no final product answers or accuracy scores. Predictions use each customer's final allowed prior history. The original logistic scaler/weights and tree model are frozen, and the weighted rule is unchanged. Save/check the lists first; scoring is a separate step. Don't rerun preparation casually: the script refuses to overwrite its frozen artifact.


In [ ]:
# Read the checked final prediction report, without opening final answer products.
from pathlib import Path
import json
import pandas as pd
project_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / '02_Data/02_Prepared/evaluation_plan.csv').exists())
final_predictions = json.loads((project_root / '05_Results/03_Checks/final_prediction_preparation.json').read_text(encoding='utf-8'))
assert final_predictions['status'] == 'passed'
assert final_predictions['final_answers_read'] is False
assert final_predictions['final_scores_generated'] is False
display(pd.DataFrame([{'Method': method, **values}
                     for method, values in final_predictions['methods_summary'].items()]))


## Final held-back basket results: saved evaluation

These are final results, separate from the earlier validation sections. Frozen lists/models and the selected weighted default didn't change. Source/plan/prediction fingerprints and overlap checks passed. The final holdout is now inspected and must not become tuning data or a supposedly fresh rerun. This cell reads saved aggregates without opening raw answers or scoring again.


In [ ]:
# Read the checked final aggregate report; do not rerun the final evaluation.
from pathlib import Path
import json
import pandas as pd
project_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / '02_Data/02_Prepared/evaluation_plan.csv').exists())
final_report = json.loads((project_root / '05_Results/03_Checks/final_evaluation.json').read_text(encoding='utf-8'))
assert final_report['status'] == 'passed'
assert final_report['models_refitted'] is False
assert final_report['predictions_regenerated'] is False
rows = [{'Method': method, 'Customers': values['customers'],
         'Precision (%)': 100 * values['precision_at_5'], 'Recall (%)': 100 * values['recall_at_5'],
         'Hit rate (%)': 100 * values['hit_at_5']}
        for method, values in final_report['overall'].items()]
display(pd.DataFrame(rows))
